# CLD Analysis — Data Processing  *(template)*

This notebook takes the **chord-length results** produced by the companion
`CLD_from_Binary_Masks` notebook (saved as `.npy` files, one per replicate),
merges them, converts pixels → microns, runs statistics, and saves tidy
result files for plotting.

**No plots are made here** — use the companion plotting notebook to visualise.

---

### How to use this notebook (for non-coders)
1. Run the cells **from top to bottom** (Shift + Enter runs the selected cell).
2. You only ever need to edit **one cell**: the one marked
   **"EDIT THIS CELL"** (Section 1). Everything else runs automatically.
3. Lines you must change are marked with `# <-- CHANGE THIS`.

### Data layout this notebook expects
- **Sample** (e.g. T1, T2, T4, 12nt)
  - **Independent replicate experiment** (r_1, r_2, …) → one `.npy` file each
    - **FOVs** (fields of view = technical replicates) inside each experiment


## Step 0 — Load the Python tools

This just imports the libraries used below (NumPy for maths, pandas for tables,
SciPy for statistics). You don't need to change anything here — just run it.

*First time only:* if you get a `ModuleNotFoundError`, install the packages by
running `pip install numpy pandas scipy` in a terminal (or in a cell with a
leading `!`).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
from pathlib import Path

## Step 1 — Configuration  ✏️ **EDIT THIS CELL**

This is the **only** cell you need to change for each new experiment. Update the
paths, the file-name pattern, your sample labels, and the pixel→micron scale,
then run the rest of the notebook unchanged.

In [ ]:
# === EDIT THIS CELL ====================================================

# Folder that CONTAINS the .npy chord-length files (output of the
# 'CLD_from_Binary_Masks' notebook). Use the full/absolute path.
data_dir = Path("PASTE_YOUR_INPUT_FOLDER_HERE")        # <-- CHANGE THIS
# Example (original): Path(".../Mg_Na/Pooled_Masks_for_CLD")

# Folder where the processed results will be saved (created if missing).
output_dir = Path("PASTE_YOUR_OUTPUT_FOLDER_HERE")     # <-- CHANGE THIS
# Example (original): Path("../Fig1_Radius_Measurement/Mg_Na/Processed_Python/CLD")
output_dir.mkdir(parents=True, exist_ok=True)

# Prefix added to every saved file, so you can tell experiments apart.
prefix = "MyExperiment_CLD"                            # <-- CHANGE THIS

# Pattern that matches your .npy files. The '*' is a wildcard.
# e.g. 'Radius_Na_r*.npy' matches Radius_Na_r1.npy, Radius_Na_r2.npy, ...
npy_glob = "Radius_*_r*.npy"                            # <-- CHANGE THIS
# The part of the file name to strip away to get a short experiment ID.
npy_stem_prefix = "Radius_Sample_r"                    # <-- CHANGE THIS

# Your sample names. ORDER MATTERS (it is reused by the plotting notebook).
dataset_labels = ["T1", "T2", "T4", "12nt"]            # <-- CHANGE THIS

# Pixel-to-micron conversion for YOUR microscope/objective.
# chord-length-in-microns = chord-in-pixels / (px_um_conv / binning)
px_um_conv = 6.1538                                    # <-- CHANGE THIS
binning = 1                                            # <-- CHANGE THIS
pxum_conv_bin = px_um_conv / binning

# P-value threshold for calling a difference 'significant'.
alpha = 0.05
# ========================================================================

## Step 2 — Load every replicate file and convert pixels → microns

Finds all `.npy` files that match your pattern, reads the chord lengths from
each, **combines the horizontal (X) and vertical (Y) chords**, and converts them
from pixels to microns. The result is organised as
`chords_by_exp[sample][experiment][FOV]`. A short summary is printed so you can
sanity-check that the expected files and counts were found.

In [ ]:
npy_files = sorted(data_dir.glob(npy_glob))
print(f'Found {len(npy_files)} replicate file(s):')
for f in npy_files:
    print(f'  {f.name}')

# Structure:
#   chords_by_exp[sample][experiment] = {fov: array of chord lengths in micron}
chords_by_exp = {ds: {} for ds in dataset_labels}

for fpath in npy_files:
    exp_id = fpath.stem.replace(npy_stem_prefix, '')  # e.g. '1', '2'
    cld_raw = np.load(fpath, allow_pickle=True).item()

    for sample in cld_raw.keys():
        if sample not in chords_by_exp:
            chords_by_exp[sample] = {}

        fov_chords = {}
        for fov, fov_data in cld_raw[sample].items():
            chords_xy = (1 / pxum_conv_bin) * np.concatenate((
                fov_data['count_x'],
                fov_data['count_y']
            ))
            fov_chords[fov] = chords_xy

        chords_by_exp[sample][exp_id] = fov_chords

# Summary
for sample in dataset_labels:
    for exp_id, fov_dict in sorted(chords_by_exp[sample].items()):
        n_chords = sum(len(v) for v in fov_dict.values())
        print(f'{sample} experiment r_{exp_id}: {len(fov_dict)} FOVs, {n_chords} total chords')

## Step 3 — Average chord length per field of view (FOV)

For each FOV, takes the **mean** chord length. These per-FOV means are the unit
of replication used in the statistics below (one number per image), which is
more conservative than treating every individual chord as independent.

In [ ]:
# fov_means[sample][experiment] = array of per-FOV mean chord lengths
fov_means = {ds: {} for ds in dataset_labels}

for sample in dataset_labels:
    for exp_id, fov_dict in sorted(chords_by_exp[sample].items()):
        means = np.array([np.mean(chords) for chords in fov_dict.values()])
        fov_means[sample][exp_id] = means
        print(f'{sample} r_{exp_id}: per-FOV means (um) = '
              f'{np.array2string(means, precision=2)}, '
              f'experiment mean = {means.mean():.2f} um')
    print()

## Step 4 — Are the independent experiments consistent? Then pool them

Before combining replicate experiments, this checks they agree:
- **2 experiments** → a two-sample t-test.
- **3+ experiments** → a one-way ANOVA.
- **1 experiment** → nothing to test.

If `p ≥ alpha` the experiments are statistically indistinguishable and are pooled
(this is the desired outcome). If `p < alpha` they differ and the cell still
pools them **but flags a warning** so you can decide what to do. It then reports
the pooled grand mean ± standard deviation.

In [ ]:
pooled_fov_means = {}  # {sample: pooled array of per-FOV means}
pooled_chords = {}     # {sample: pooled array of all individual chords}
grand_mean = {}        # {sample: grand mean}
grand_std = {}         # {sample: SD of pooled FOV means}
ttest_results = {}     # {sample: {t, p}}
pool_status = {}       # {sample: bool}

for sample in dataset_labels:
    experiments = sorted(fov_means[sample].keys())
    print(f'--- {sample} ---')

    if len(experiments) == 2:
        e1, e2 = experiments
        t_stat, p_val = stats.ttest_ind(fov_means[sample][e1], fov_means[sample][e2])
        ttest_results[sample] = {'t': t_stat, 'p': p_val}
        sig = 'SIGNIFICANT' if p_val < alpha else 'not significant'
        print(f'  r_{e1} (n={len(fov_means[sample][e1])}) vs '
              f'r_{e2} (n={len(fov_means[sample][e2])}):  '
              f't = {t_stat:.3f}, p = {p_val:.4f} ({sig})')
        pool_status[sample] = p_val >= alpha

        if p_val >= alpha:
            print(f'  \u2713 Experiments consistent \u2192 pooling')
        else:
            print(f'  \u26a0 Experiments differ! Pooling anyway but flagged.')

    elif len(experiments) == 1:
        print(f'  Only one experiment, no t-test needed.')
        pool_status[sample] = True

    else:
        groups = [fov_means[sample][e] for e in experiments]
        f_stat, p_val = stats.f_oneway(*groups)
        ttest_results[sample] = {'F': f_stat, 'p': p_val}
        sig = 'SIGNIFICANT' if p_val < alpha else 'not significant'
        print(f'  ANOVA across {len(experiments)} experiments: '
              f'F = {f_stat:.3f}, p = {p_val:.4f} ({sig})')
        pool_status[sample] = p_val >= alpha

    # Pool FOV means and all individual chords
    pooled_fov_means[sample] = np.concatenate(
        [fov_means[sample][e] for e in experiments])
    pooled_chords[sample] = np.concatenate([
        chord for e in experiments
        for chord in chords_by_exp[sample][e].values()
    ])

    grand_mean[sample] = pooled_fov_means[sample].mean()
    grand_std[sample] = pooled_fov_means[sample].std(ddof=1)

    print(f'  Grand mean = {grand_mean[sample]:.2f} \u00b1 {grand_std[sample]:.2f} \u00b5m (SD, '
          f'n = {len(pooled_fov_means[sample])} FOVs)\n')

## Step 5 — Compare samples against each other

For every pair of samples this runs two complementary tests:
- a **t-test** on the per-FOV means (do the *averages* differ?), and
- a **Kolmogorov–Smirnov (KS) test** on all individual chords (do the whole
  *distributions* differ in shape?).

Significance stars: `*` p<0.05, `**` p<0.01, `***` p<0.001, `n.s.` not
significant. Results are collected into a table.

In [ ]:
comparison_rows = []

for s1, s2 in combinations(dataset_labels, 2):
    t_stat, t_p = stats.ttest_ind(pooled_fov_means[s1], pooled_fov_means[s2])
    ks_stat, ks_p = stats.ks_2samp(pooled_chords[s1], pooled_chords[s2])

    comparison_rows.append({
        'Comparison': f'{s1} vs {s2}',
        't-stat': t_stat,
        't-test p': t_p,
        't-test sig.': '***' if t_p < 0.001 else '**' if t_p < 0.01 else '*' if t_p < 0.05 else 'n.s.',
        'KS-stat': ks_stat,
        'KS p': ks_p,
        'KS sig.': '***' if ks_p < 0.001 else '**' if ks_p < 0.01 else '*' if ks_p < 0.05 else 'n.s.',
    })

    print(f'{s1} vs {s2}:')
    print(f'  t-test (FOV means):       t = {t_stat:.3f}, p = {t_p:.2e}  ({comparison_rows[-1]["t-test sig."]})')
    print(f'  KS test (full distrib.): KS = {ks_stat:.4f}, p = {ks_p:.2e}  ({comparison_rows[-1]["KS sig."]})')
    print()

df_comparisons = pd.DataFrame(comparison_rows)
display(df_comparisons)

## Step 6 — Build summary tables

Two tidy tables: one **per experiment** (mean ± SD chord length and FOV count for
each replicate) and one **per sample** (the pooled grand mean, whether pooling
was justified, and the consistency p-value). These are what you'll save next.

In [ ]:
# Per-experiment summary
rows = []
for sample in dataset_labels:
    for exp_id in sorted(fov_means[sample].keys()):
        m = fov_means[sample][exp_id]
        rows.append({
            'Sample': sample,
            'Experiment': f'r_{exp_id}',
            'N_FOVs': len(m),
            'Mean_CLD_um': m.mean(),
            'SD_CLD_um': m.std(ddof=1),
        })

df_exp = pd.DataFrame(rows)
display(df_exp)

# Grand mean summary
rows_grand = []
for sample in dataset_labels:
    rows_grand.append({
        'Sample': sample,
        'Grand_mean_um': grand_mean[sample],
        'SD_um': grand_std[sample],
        'N_FOVs_total': len(pooled_fov_means[sample]),
        'Pooled': pool_status[sample],
        't-test p': ttest_results.get(sample, {}).get('p', np.nan),
    })

df_grand = pd.DataFrame(rows_grand)
display(df_grand)

## Step 7 — Save everything for the plotting notebook

Writes the processed data to your `output_dir`:
- **`.npy` files** — the dictionaries (chords, per-FOV means, pooled values,
  grand means/SDs) that the plotting notebook reads back in.
- **`.csv` files** — the human-readable summary tables (open these in Excel).

All file names start with your `prefix` so nothing gets overwritten between
experiments.

In [ ]:
# --- Save dictionaries as .npy ---
np.save(output_dir / f'{prefix}_chords_by_exp.npy', chords_by_exp, allow_pickle=True)
np.save(output_dir / f'{prefix}_fov_means.npy', fov_means, allow_pickle=True)
np.save(output_dir / f'{prefix}_pooled_fov_means.npy', pooled_fov_means, allow_pickle=True)
np.save(output_dir / f'{prefix}_pooled_chords.npy', pooled_chords, allow_pickle=True)
np.save(output_dir / f'{prefix}_grand_mean.npy', grand_mean, allow_pickle=True)
np.save(output_dir / f'{prefix}_grand_std.npy', grand_std, allow_pickle=True)

print('Saved .npy files:')
for f in sorted(output_dir.glob(f'{prefix}_*.npy')):
    print(f'  {f.name}')

# --- Save CSVs ---
df_exp.to_csv(output_dir / f'{prefix}_experiment_summary.csv', index=False)
df_grand.to_csv(output_dir / f'{prefix}_grand_summary.csv', index=False)
df_comparisons.to_csv(output_dir / f'{prefix}_pairwise_comparisons.csv', index=False)
print(f'\nSaved 3 CSV files')